In [3]:
!pip install streamlit scikit-learn pandas numpy matplotlib seaborn plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [4]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/lab data sml/Online Shopper Dataset.csv')

print("Dataset Shape:", df.shape)
display(df.head())

print("\nDataset Information:")
df.info()

Dataset Shape: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print("Missing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nColumn Names:")
print(df.columns.tolist())

Missing Values:


,0
Administrative,0
Administrative_Duration,0
Informational,0
Informational_Duration,0
ProductRelated,0
ProductRelated_Duration,0
BounceRates,0
ExitRates,0
PageValues,0
SpecialDay,0



Duplicate Rows: 125

Column Names:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Convert Revenue into integer (0 and 1)
df['Revenue'] = df['Revenue'].astype(int)

print("Dataset Shape After Cleaning:", df.shape)

print("\nRevenue Distribution:")
print(df['Revenue'].value_counts())

print("\nData Cleaning Completed!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))

sns.countplot(data=df, x='Revenue')

plt.title('Purchase vs Non-Purchase Sessions')
plt.xlabel('Revenue (0 = No Purchase, 1 = Purchase)')
plt.ylabel('Number of Sessions')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(data=df, x='Revenue', y='PageValues')

plt.title('Page Values vs Purchase Behaviour')
plt.xlabel('Revenue (0 = No Purchase, 1 = Purchase)')
plt.ylabel('Page Values')

plt.show()

In [ ]:
plt.figure(figsize=(14, 9))

numeric_df = df.select_dtypes(include=['int64', 'float64'])

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Separate input features and target
X = df.drop('Revenue', axis=1)
y = df['Revenue']

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(exclude=['int64', 'float64']).columns

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both preprocessors
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)
print("Numerical Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create the Logistic Regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression Model Trained Successfully!")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Remove Revenue from clustering data
X_cluster = df.drop('Revenue', axis=1)

# Identify numerical and categorical features
num_cols = X_cluster.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_cluster.select_dtypes(exclude=['int64', 'float64']).columns

# Numerical preprocessing
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
cluster_preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Transform clustering data
X_cluster_scaled = cluster_preprocessor.fit_transform(X_cluster)

print("Clustering Data Shape:", X_cluster_scaled.shape)
print("Clustering Preprocessing Completed!")

In [ ]:
inertia = []
silhouette_scores = []
k_values = range(2, 7)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_cluster_scaled)

    inertia.append(kmeans.inertia_)
    silhouette_scores.append(
        silhouette_score(
            X_cluster_scaled,
            labels,
            sample_size=3000,
            random_state=42
        )
    )

# Plot Elbow Method
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

# Plot Silhouette Scores
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker='o')
plt.title('Silhouette Score Analysis')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.show()

print("K Values:", list(k_values))
print("Silhouette Scores:", silhouette_scores)

In [ ]:
# Create final K-Means model
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

# Assign clusters
df['Cluster'] = kmeans.fit_predict(X_cluster_scaled)

print("K-Means Clustering Completed!")
print("\nCluster Distribution:")
print(df['Cluster'].value_counts())

In [ ]:
cluster_profile = df.groupby('Cluster').agg(
    Total_Sessions=('Revenue', 'count'),
    Purchase_Rate=('Revenue', 'mean'),
    Average_Page_Values=('PageValues', 'mean'),
    Average_Administrative=('Administrative', 'mean'),
    Average_Product_Related=('ProductRelated', 'mean')
).reset_index()

cluster_profile['Purchase_Rate'] *= 100

display(cluster_profile.round(2))

In [ ]:
import joblib
import json

# Save trained models and preprocessors
joblib.dump(model, 'purchase_model.pkl')
joblib.dump(kmeans, 'clustering_model.pkl')
joblib.dump(cluster_preprocessor, 'cluster_preprocessor.pkl')

# Save dataset with cluster labels
df.to_csv('customer_data.csv', index=False)

# Save testing data
test_data = X_test.copy()
test_data['Revenue'] = y_test
test_data.to_csv('test_data.csv', index=False)

# Save evaluation metrics
metrics = {
    "Accuracy": float(accuracy_score(y_test, y_pred)),
    "Precision": float(precision_score(y_test, y_pred)),
    "Recall": float(recall_score(y_test, y_pred)),
    "F1 Score": float(f1_score(y_test, y_pred)),
    "ROC-AUC": float(roc_auc_score(y_test, y_prob))
}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("All project files saved successfully!")

In [ ]:
import os
import zipfile

files = [
    'purchase_model.pkl',
    'clustering_model.pkl',
    'cluster_preprocessor.pkl',
    'customer_data.csv',
    'test_data.csv',
    'model_metrics.json'
]

with zipfile.ZipFile('project_files.zip', 'w') as zipf:
    for file in files:
        zipf.write(file)

print("Project files packaged successfully!")
print("Files included:", files)

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import json
import plotly.express as px

# Page configuration
st.set_page_config(
    page_title="Shopper Intelligence",
    page_icon="🛍️",
    layout="wide"
)

# Load project files
@st.cache_resource
def load_models():
    purchase_model = joblib.load("purchase_model.pkl")
    clustering_model = joblib.load("clustering_model.pkl")
    cluster_preprocessor = joblib.load("cluster_preprocessor.pkl")
    return purchase_model, clustering_model, cluster_preprocessor

@st.cache_data
def load_data():
    return pd.read_csv("customer_data.csv")

model, kmeans, cluster_preprocessor = load_models()
df = load_data()

with open("model_metrics.json", "r") as f:
    metrics = json.load(f)

# Custom styling
st.markdown("""
<style>
.main {
    background-color: #f8fafc;
}
h1, h2, h3 {
    color: #172554;
}
[data-testid="stMetric"] {
    background-color: white;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #e2e8f0;
}
</style>
""", unsafe_allow_html=True)

# Sidebar navigation
st.sidebar.title("🛍️ Shopper Intelligence")
st.sidebar.caption("Customer Segmentation & Purchase Prediction")

page = st.sidebar.radio(
    "Navigation",
    [
        "Dashboard",
        "Customer Segmentation",
        "Purchase Prediction",
        "Model Performance",
        "About Project"
    ]
)

# DASHBOARD
if page == "Dashboard":

    st.title("Shopper Intelligence Dashboard")
    st.write("Understand shopper behaviour and predict purchase intentions.")

    total_sessions = len(df)
    purchases = int(df["Revenue"].sum())
    non_purchases = total_sessions - purchases
    conversion_rate = purchases / total_sessions * 100

    c1, c2, c3, c4 = st.columns(4)

    c1.metric("Total Sessions", f"{total_sessions:,}")
    c2.metric("Purchases", f"{purchases:,}")
    c3.metric("Non-Purchases", f"{non_purchases:,}")
    c4.metric("Conversion Rate", f"{conversion_rate:.2f}%")

    st.subheader("Purchase Distribution")

    fig = px.pie(
        names=["Purchase", "No Purchase"],
        values=[purchases, non_purchases],
        hole=0.55
    )
    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Project Overview")
    st.write("""
    This project uses Machine Learning to analyse online shopping sessions.

    - Logistic Regression predicts purchase intention.
    - K-Means identifies groups of similar shopping sessions.
    - Model evaluation measures prediction performance.
    """)

# CUSTOMER SEGMENTATION
elif page == "Customer Segmentation":

    st.title("Customer Segmentation")
    st.write("Explore shopping sessions grouped by similar behaviour.")

    cluster_counts = df["Cluster"].value_counts().sort_index()

    c1, c2 = st.columns(2)

    c1.metric("Number of Segments", df["Cluster"].nunique())
    c2.metric("Total Sessions Analysed", f"{len(df):,}")

    fig = px.bar(
        x=cluster_counts.index.astype(str),
        y=cluster_counts.values,
        labels={"x": "Cluster", "y": "Sessions"},
        title="Sessions in Each Cluster"
    )
    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Cluster Profiling")

    profile = df.groupby("Cluster").agg(
        Total_Sessions=("Revenue", "count"),
        Purchase_Rate=("Revenue", "mean"),
        Average_Page_Values=("PageValues", "mean"),
        Average_Product_Related=("ProductRelated", "mean")
    ).reset_index()

    profile["Purchase_Rate"] *= 100

    st.dataframe(
        profile.round(2),
        use_container_width=True,
        hide_index=True
    )

    fig = px.bar(
        profile,
        x="Cluster",
        y="Purchase_Rate",
        color="Cluster",
        title="Purchase Rate by Cluster",
        labels={"Purchase_Rate": "Purchase Rate (%)"}
    )
    st.plotly_chart(fig, use_container_width=True)

# PURCHASE PREDICTION
elif page == "Purchase Prediction":

    st.title("Purchase Intention Prediction")
    st.write("Enter shopping session details to predict purchase intention.")

    input_data = {}

    with st.form("prediction_form"):

        for column in model.named_steps["preprocessor"].feature_names_in_:
            if column in df.columns:

                if pd.api.types.is_numeric_dtype(df[column]):
                    value = float(df[column].median())

                    if pd.api.types.is_integer_dtype(df[column]):
                        input_data[column] = st.number_input(
                            column,
                            value=int(value)
                        )
                    else:
                        input_data[column] = st.number_input(
                            column,
                            value=value
                        )

                else:
                    options = df[column].dropna().unique().tolist()

                    if len(options) > 0:
                        input_data[column] = st.selectbox(
                            column,
                            options
                        )

        submitted = st.form_submit_button("Predict Purchase")

    if submitted:
        input_df = pd.DataFrame([input_data])
        prediction = model.predict(input_df)[0]
        probability = model.predict_proba(input_df)[0][1]

        st.subheader("Prediction Result")

        if prediction == 1:
            st.success("The model predicts a purchase!")
        else:
            st.info("The model predicts no purchase.")

        st.metric("Purchase Probability", f"{probability * 100:.2f}%")
        st.progress(float(probability))

# MODEL PERFORMANCE
elif page == "Model Performance":

    st.title("Model Performance")
    st.write("Evaluation results of the Logistic Regression model.")

    c1, c2, c3 = st.columns(3)

    c1.metric("Accuracy", f"{metrics['Accuracy'] * 100:.2f}%")
    c2.metric("Precision", f"{metrics['Precision'] * 100:.2f}%")
    c3.metric("Recall", f"{metrics['Recall'] * 100:.2f}%")

    c4, c5 = st.columns(2)

    c4.metric("F1 Score", f"{metrics['F1 Score'] * 100:.2f}%")
    c5.metric("ROC-AUC", f"{metrics['ROC-AUC'] * 100:.2f}%")

    fig = px.bar(
        x=list(metrics.keys()),
        y=[value * 100 for value in metrics.values()],
        labels={"x": "Metric", "y": "Score (%)"},
        title="Model Evaluation Metrics",
        range_y=[0, 100]
    )

    st.plotly_chart(fig, use_container_width=True)

# ABOUT PROJECT
elif page == "About Project":

    st.title("About the Project")

    st.write("""
    ## Customer Segmentation and Purchase Prediction

    This project analyses online shopping sessions using Machine Learning.

    ### Technologies Used
    - Python
    - Pandas and NumPy
    - Scikit-learn
    - Streamlit
    - Plotly

    ### Machine Learning Algorithms

    **1. Logistic Regression**

    Used to predict whether an online shopping session will result in a purchase.

    **2. K-Means Clustering**

    Used to group shopping sessions according to similar browsing behaviour.

    ### Dataset

    Online Shoppers Purchasing Intention Dataset.

    ### Project Objective

    To understand online shopper behaviour, identify meaningful shopping
    segments and predict purchase intention using Machine Learning.
    """)

In [ ]:
import os

required_files = [
    "app.py",
    "purchase_model.pkl",
    "clustering_model.pkl",
    "cluster_preprocessor.pkl",
    "customer_data.csv",
    "model_metrics.json"
]

for file in required_files:
    if os.path.exists(file):
        print("✅", file, "found")
    else:
        print("❌", file, "missing")

In [ ]:
import subprocess
import time
import requests

# Start Streamlit
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Check whether the server starts
for i in range(20):
    try:
        response = requests.get("http://127.0.0.1:8501", timeout=2)
        if response.status_code == 200:
            print("Website server started successfully!")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    print("Server did not start. Check streamlit.log:")
    print(open("streamlit.log").read()[-3000:])

In [ ]:
import os
import subprocess
import time

# Download Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared

# Give execution permission
!chmod +x cloudflared

# Start the tunnel
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=open("cloudflared.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for the public URL
for i in range(30):
    time.sleep(2)

    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log") as f:
            log = f.read()

        import re
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)

        if match:
            print("YOUR WEBSITE LINK:")
            print(match.group(0))
            break
else:
    print("Tunnel link not found. Log:")
    print(log[-3000:])

In [ ]:
import os

os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/config.toml", "w") as f:
    f.write("""
[theme]
base="light"
primaryColor="#6C63FF"
backgroundColor="#FFFFFF"
secondaryBackgroundColor="#F4F5FB"
textColor="#20243A"
font="sans serif"
""")

print("Professional light theme configured!")

In [ ]:

%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import json
import plotly.express as px

st.set_page_config(
    page_title="Shopper Intelligence",
    page_icon="🛍️",
    layout="wide",
    initial_sidebar_state="expanded"
)

@st.cache_resource
def load_models():
    return (
        joblib.load("purchase_model.pkl"),
        joblib.load("clustering_model.pkl"),
        joblib.load("cluster_preprocessor.pkl")
    )

@st.cache_data
def load_data():
    return pd.read_csv("customer_data.csv")

model, kmeans, cluster_preprocessor = load_models()
df = load_data()

with open("model_metrics.json") as f:
    metrics = json.load(f)

# PROFESSIONAL LIGHT THEME
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

html, body, [class*="css"] {
    font-family: 'Inter', sans-serif;
}

.stApp {
    background: #F7F8FC;
    color: #20243A;
}

[data-testid="stSidebar"] {
    background: #FFFFFF;
    border-right: 1px solid #E8EAF2;
}

[data-testid="stSidebar"] h1 {
    color: #5145CD;
    font-weight: 800;
}

h1, h2, h3 {
    color: #20243A !important;
    font-weight: 750 !important;
}

p, label, span {
    color: #454B63;
}

.hero {
    background: linear-gradient(120deg, #5145CD, #8175F5);
    padding: 32px;
    border-radius: 20px;
    color: white;
    margin-bottom: 25px;
}

.hero h1, .hero p {
    color: white !important;
}

.metric-card {
    background: white;
    border: 1px solid #E8EAF2;
    padding: 22px;
    border-radius: 16px;
    box-shadow: 0 4px 15px rgba(30,40,90,0.04);
    min-height: 115px;
}

.metric-label {
    color: #737991;
    font-size: 14px;
    font-weight: 600;
}

.metric-value {
    color: #5145CD;
    font-size: 29px;
    font-weight: 800;
    margin-top: 8px;
}

.brand-card {
    background: white;
    border: 1px solid #E8EAF2;
    border-radius: 13px;
    padding: 16px 8px;
    text-align: center;
    font-size: 17px;
    font-weight: 800;
}

.section-heading {
    font-size: 21px;
    font-weight: 750;
    color: #20243A;
    margin-top: 24px;
    margin-bottom: 15px;
}

div[data-testid="stForm"] {
    background: white;
    padding: 22px;
    border-radius: 16px;
    border: 1px solid #E8EAF2;
}

.stButton button,
.stFormSubmitButton button {
    background: #5145CD;
    color: white;
    border-radius: 10px;
    border: none;
    font-weight: 700;
}

.stButton button:hover,
.stFormSubmitButton button:hover {
    background: #3930A6;
    color: white;
}

[data-testid="stMetric"] {
    background: white;
    border: 1px solid #E8EAF2;
    padding: 18px;
    border-radius: 14px;
}

[data-testid="stMetricLabel"] {
    color: #626A82 !important;
}

[data-testid="stMetricValue"] {
    color: #5145CD !important;
}

</style>
""", unsafe_allow_html=True)


# SIDEBAR
st.sidebar.markdown("# 🛍️ Shopper IQ")
st.sidebar.caption("CUSTOMER ANALYTICS PLATFORM")
st.sidebar.markdown("---")

page = st.sidebar.radio(
    "EXPLORE",
    [
        "Dashboard",
        "Customer Segmentation",
        "Purchase Prediction",
        "Model Performance",
        "About Project"
    ]
)

st.sidebar.markdown("---")
st.sidebar.caption("Powered by Machine Learning")


# HELPER FOR METRIC CARDS
def metric_card(label, value):
    st.markdown(
        f"""
        <div class="metric-card">
            <div class="metric-label">{label}</div>
            <div class="metric-value">{value}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


# DASHBOARD
if page == "Dashboard":

    st.markdown("""
    <div class="hero">
        <h1>Shopper Intelligence Dashboard</h1>
        <p>Discover shopping patterns. Understand customers.
        Predict purchase intentions.</p>
    </div>
    """, unsafe_allow_html=True)

    total = len(df)
    purchases = int(df["Revenue"].sum())
    no_purchases = total - purchases
    conversion = purchases / total * 100

    c1, c2, c3, c4 = st.columns(4)

    with c1:
        metric_card("🛒 Total Sessions", f"{total:,}")
    with c2:
        metric_card("💜 Purchases", f"{purchases:,}")
    with c3:
        metric_card("👥 No Purchases", f"{no_purchases:,}")
    with c4:
        metric_card("📈 Conversion Rate", f"{conversion:.2f}%")

    st.markdown('<div class="section-heading">Shopping Platform Inspiration</div>',
                unsafe_allow_html=True)

    brands = st.columns(4)
    brand_names = [
        ("amazon", "Amazon", "#232F3E"),
        ("flipkart", "Flipkart", "#2874F0"),
        ("nykaa", "NYKAA", "#E91E63"),
        ("myntra", "Myntra", "#E83E8C")
    ]

    for col, (icon, name, color) in zip(brands, brand_names):
        with col:
            st.markdown(
                f'<div class="brand-card" style="color:{color}">'
                f'🛍️ {name}</div>',
                unsafe_allow_html=True
            )

    left, right = st.columns(2)

    with left:
        st.markdown('<div class="section-heading">Purchase Distribution</div>',
                    unsafe_allow_html=True)

        fig = px.pie(
            names=["Purchase", "No Purchase"],
            values=[purchases, no_purchases],
            hole=0.62,
            color_discrete_sequence=["#7568E8", "#C9C5F5"]
        )
        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",
            font_color="#343A55"
        )
        st.plotly_chart(fig, use_container_width=True)

    with right:
        st.markdown('<div class="section-heading">Session Overview</div>',
                    unsafe_allow_html=True)

        fig = px.bar(
            x=["Purchase", "No Purchase"],
            y=[purchases, no_purchases],
            color=["Purchase", "No Purchase"],
            color_discrete_sequence=["#7568E8", "#B7B1EF"],
            labels={"x": "Session Type", "y": "Number of Sessions"}
        )
        fig.update_layout(
            showlegend=False,
            paper_bgcolor="rgba(0,0,0,0)",
            font_color="#343A55"
        )
        st.plotly_chart(fig, use_container_width=True)

    st.info("This dashboard analyses online shopping sessions using Machine Learning.")


# CUSTOMER SEGMENTATION
elif page == "Customer Segmentation":

    st.markdown('<div class="hero"><h1>Customer Segmentation</h1>'
                '<p>Explore groups of similar online shopping sessions.</p></div>',
                unsafe_allow_html=True)

    counts = df["Cluster"].value_counts().sort_index()

    c1, c2 = st.columns(2)
    with c1:
        metric_card("🧩 Total Segments", "2")
    with c2:
        metric_card("👥 Sessions Analysed", f"{len(df):,}")

    fig = px.bar(
        x=counts.index.astype(str),
        y=counts.values,
        color=counts.index.astype(str),
        color_discrete_sequence=["#7568E8", "#B7B1EF"],
        labels={"x": "Cluster", "y": "Sessions"},
        title="Shopping Sessions by Cluster"
    )
    st.plotly_chart(fig, use_container_width=True)

    profile = df.groupby("Cluster").agg(
        Total_Sessions=("Revenue", "count"),
        Purchase_Rate=("Revenue", "mean"),
        Average_Page_Values=("PageValues", "mean"),
        Average_Product_Related=("ProductRelated", "mean")
    ).reset_index()

    profile["Purchase_Rate"] *= 100

    st.markdown('<div class="section-heading">Cluster Profiles</div>',
                unsafe_allow_html=True)
    st.dataframe(profile.round(2), use_container_width=True, hide_index=True)

    fig = px.bar(
        profile,
        x="Cluster",
        y="Purchase_Rate",
        color="Cluster",
        color_discrete_sequence=["#7568E8", "#B7B1EF"],
        title="Purchase Rate by Segment"
    )
    st.plotly_chart(fig, use_container_width=True)

    st.caption("Clusters are groups of shopping sessions, not identified individual customers.")


# PURCHASE PREDICTION
elif page == "Purchase Prediction":

    st.markdown('<div class="hero"><h1>Purchase Prediction</h1>'
                '<p>Enter shopping session details to estimate purchase intention.</p></div>',
                unsafe_allow_html=True)

    input_data = {}

    with st.form("prediction_form"):

        st.subheader("Shopping Session Details")

        columns = st.columns(2)
        feature_names = model.named_steps["preprocessor"].feature_names_in_

        for i, column in enumerate(feature_names):
            if column not in df.columns:
                continue

            with columns[i % 2]:
                if pd.api.types.is_numeric_dtype(df[column]):
                    median = float(df[column].median())

                    if pd.api.types.is_integer_dtype(df[column]):
                        input_data[column] = st.number_input(
                            column, value=int(median)
                        )
                    else:
                        input_data[column] = st.number_input(
                            column, value=median
                        )
                else:
                    options = df[column].dropna().unique().tolist()
                    input_data[column] = st.selectbox(column, options)

        submitted = st.form_submit_button("🔮 Predict Purchase", use_container_width=True)

    if submitted:
        input_df = pd.DataFrame([input_data])
        prediction = model.predict(input_df)[0]
        probability = model.predict_proba(input_df)[0][1]

        st.markdown('<div class="section-heading">Prediction Result</div>',
                    unsafe_allow_html=True)

        if prediction == 1:
            st.success("🛍️ The model predicts that this session may result in a purchase.")
        else:
            st.warning("The model predicts that this session may not result in a purchase.")

        st.metric("Purchase Probability", f"{probability * 100:.2f}%")
        st.progress(float(probability))


# MODEL PERFORMANCE
elif page == "Model Performance":

    st.markdown('<div class="hero"><h1>Model Performance</h1>'
                '<p>Evaluate how well the purchase prediction model performs.</p></div>',
                unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)

    values = list(metrics.items())

    for col, (name, value) in zip([c1, c2, c3], values[:3]):
        with col:
            metric_card(name, f"{value * 100:.2f}%")

    c4, c5 = st.columns(2)

    for col, (name, value) in zip([c4, c5], values[3:]):
        with col:
            metric_card(name, f"{value * 100:.2f}%")

    fig = px.bar(
        x=list(metrics.keys()),
        y=[v * 100 for v in metrics.values()],
        color=list(metrics.keys()),
        color_discrete_sequence=["#7568E8", "#A59CF0", "#C3BDF7"],
        labels={"x": "Evaluation Metric", "y": "Score (%)"},
        title="Model Evaluation Results",
        range_y=[0, 100]
    )
    st.plotly_chart(fig, use_container_width=True)


# ABOUT PROJECT
elif page == "About Project":

    st.markdown('<div class="hero"><h1>About Shopper Intelligence</h1>'
                '<p>Machine Learning for understanding online shopper behaviour.</p></div>',
                unsafe_allow_html=True)

    st.subheader("Project Objective")

    st.write("""
    Customer Segmentation and Purchase Prediction uses the Online Shoppers
    Purchasing Intention dataset to analyse shopping sessions and predict
    whether a session will result in a purchase.
    """)

    st.subheader("Technologies Used")

    st.markdown("""
    - Python
    - Pandas and NumPy
    - Scikit-learn
    - Streamlit
    - Plotly
    """)

    st.subheader("Machine Learning Algorithms")

    st.markdown("""
    **Logistic Regression:** Predicts purchase intention.

    **K-Means Clustering:** Groups similar shopping sessions based on browsing behaviour.
    """)

    st.subheader("Project Workflow")

    st.markdown("""
    1. Data Collection
    2. Data Cleaning
    3. Exploratory Data Analysis
    4. Data Preprocessing
    5. Purchase Prediction
    6. Model Evaluation
    7. Customer Segmentation
    8. Cluster Profiling
    """)


In [ ]:
streamlit_process.terminate()

In [ ]:
import subprocess
import time
import requests

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.address", "0.0.0.0"],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

for i in range(20):
    try:
        response = requests.get("http://127.0.0.1:8501", timeout=2)
        if response.status_code == 200:
            print("New website design is running!")
            break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    print(open("streamlit.log").read()[-3000:])

In [ ]:
# Update the website's visual styling
with open("app.py", "r") as f:
    code = f.read()

# Make hero banner text clearly white
code = code.replace(
    '<h1>Shopper Intelligence Dashboard</h1>',
    '<h1 style="color:white !important;">Shopper Intelligence Dashboard</h1>'
)

# Improve the hero banner appearance
code = code.replace(
    'padding: 32px;',
    'padding: 38px;'
)

# Improve metric cards
code = code.replace(
    'border-radius: 16px;',
    'border-radius: 20px;'
)

code = code.replace(
    'box-shadow: 0 4px 15px rgba(30,40,90,0.04);',
    'box-shadow: 0 8px 24px rgba(80,65,180,0.09);'
)

with open("app.py", "w") as f:
    f.write(code)

print("Dashboard styling updated!")

In [ ]:
with open("app.py", "r") as f:
    code = f.read()

code = code.replace(
    "h1, h2, h3 {\n    color: #20243A !important;\n    font-weight: 750 !important;\n}",
    """h1, h2, h3 {
    color: #20243A !important;
    font-weight: 750 !important;
}

.hero h1 {
    color: #FFFFFF !important;
}

.hero p {
    color: #FFFFFF !important;
}"""
)

with open("app.py", "w") as f:
    f.write(code)

print("Hero heading colour fixed!")

In [ ]:
with open("app.py", "r") as f:
    code = f.read()

css_fix = """
.hero h1 {
    color: #FFFFFF !important;
    -webkit-text-fill-color: #FFFFFF !important;
}

.hero p {
    color: #FFFFFF !important;
    -webkit-text-fill-color: #FFFFFF !important;
}
"""

code = code.replace("</style>", css_fix + "\n</style>")

with open("app.py", "w") as f:
    f.write(code)

print("Hero heading colour updated!")

In [ ]:

with open("app.py", "r") as f:
    code = f.read()

old_hero = '''    <div class="hero">
        <h1>Shopper Intelligence Dashboard</h1>
        <p>Discover shopping patterns. Understand customers.
        Predict purchase intentions.</p>
    </div>'''

new_hero = '''    <div class="hero hero-layout">
        <div class="hero-content">
            <h1>Shopper Intelligence Dashboard</h1>
            <p>Discover shopping patterns. Understand customers.
            Predict purchase intentions.</p>
            <div class="hero-tag">🛍️ Explore Shopping Insights →</div>
        </div>
        <div class="hero-art">
            <span>🛒</span>
            <span>🛍️</span>
            <span>💜</span>
            <span>✨</span>
        </div>
    </div>'''

if old_hero in code:
    code = code.replace(old_hero, new_hero)
else:
    print("Hero section not found. Don't overwrite app.py.")

css = """
.hero-layout {
    display: flex;
    align-items: center;
    justify-content: space-between;
    min-height: 210px;
}

.hero-content {
    max-width: 75%;
}

.hero-art {
    font-size: 43px;
    display: flex;
    flex-direction: column;
    gap: 8px;
    align-items: center;
}

.hero-tag {
    display: inline-block;
    background: white;
    color: #6246D9;
    padding: 10px 18px;
    border-radius: 25px;
    font-weight: 700;
    margin-top: 12px;
}
"""

code = code.replace("</style>", css + "\n</style>")

with open("app.py", "w") as f:
    f.write(code)

print("Shopping illustration section added!")


In [ ]:
import zipfile
from google.colab import files
import os

backup_files = [
    "app.py",
    "purchase_model.pkl",
    "clustering_model.pkl",
    "cluster_preprocessor.pkl",
    "customer_data.csv",
    "test_data.csv",
    "model_metrics.json",
    ".streamlit/config.toml"
]

with zipfile.ZipFile("Shopper_Intelligence_Backup.zip", "w") as z:
    for file in backup_files:
        if os.path.exists(file):
            z.write(file)

print("Backup created successfully!")

files.download("Shopper_Intelligence_Backup.zip")